# Fine-tuning di TrackNet sul tennis (Colab, GPU)

Questo notebook **riaddestra un po'** (fine-tuning) la rete della pallina, TrackNet, partendo dai pesi che usa già il programma (`TrackNet_best.pt`, addestrati sul **badminton**) e mostrandole le palline da **tennis** del dataset pubblico del primo TrackNet (riprese TV di 10 partite, 1280x720). Poi dà il **voto** prima/dopo sul **set di test etichettato a mano** (video mai usati per addestrare).

Il programma principale **non cambia**: i nuovi pesi finiscono su Drive in `tracknet_finetune/<RUN>/TrackNet_tennis.pt`. Usarli nel programma è il passo successivo (integrazione), da fare solo se il voto migliora.

**Preparazione, una volta sola**
- `ckpts/TrackNet_best.pt` su Drive: c'è già (lo usa il notebook principale).
- Il set di test: `etichetta_pallina.py` lo copia da solo su Drive in `tracknet_finetune/etichette_test.csv` (oppure arriva da GitHub dopo il push).
- Il dataset pubblico: la cella 5 prova a scaricarlo da sola. Se non ci riesce, apri [la cartella del dataset](https://drive.google.com/drive/folders/11r0RUaQHX7I3ANkaYG4jOxXK1OYo01Ut), clic destro sul nome della cartella → **Organizza → Aggiungi scorciatoia** → scegli `Il mio Drive/rf_coach_vision/datasets`, poi rilancia la cella 5.

**Ogni volta:** Runtime → Cambia tipo di runtime → **GPU A100** (va anche la T4, più lenta), poi le celle in ordine. La prima volta fermati dopo la **7 (prova)**: se in fondo c'è scritto `PROVA RIUSCITA`, lancia la 8.

Se Colab si disconnette durante l'addestramento: rilancia 1-6 (il dataset pronto torna dal Drive in pochi minuti) e la 8 con `RIPRENDI = True`.

## 1. Controllo GPU

In [ ]:
!nvidia-smi -L

## 2. Collega Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 3. Codice: il nostro da GitHub e quello originale di TrackNetV3

Il codice di addestramento è quello **originale** di TrackNetV3 (qaz812345/TrackNetV3, licenza MIT), fermo alla versione verificata. La cella controlla che il modello sia identico a `tracknet3/model.py` del programma: così i pesi nuovi si usano senza cambiare niente.

In [ ]:
import os, shutil, glob, subprocess, sys, json

REPO = "anrundo-2312/rf_coach_vision"
DRIVE_DIR = "/content/drive/MyDrive/rf_coach_vision"
WORK_DIR = "/content/rf_coach_vision"
FT_DRIVE = os.path.join(DRIVE_DIR, "tracknet_finetune")      # risultati del fine-tuning su Drive
TNV3_DIR = "/content/TrackNetV3"                               # codice originale di addestramento
TNV3_COMMIT = "6eda442ada1740573f200f836d93edc9a541ee86"       # versione verificata

# Il repo e' pubblico: il token serve solo se un giorno torna privato (come nel notebook principale).
token = ""
try:
    from google.colab import userdata
    token = (userdata.get("GITHUB_TOKEN") or "").strip()
except Exception:
    pass
url = f"https://{REPO.split('/')[0]}:{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"

os.chdir("/content")
if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)
r = subprocess.run(["git", "clone", "--depth", "1", url, WORK_DIR], capture_output=True, text=True,
                   env={**os.environ, "GIT_TERMINAL_PROMPT": "0"})
if r.returncode != 0:
    raise SystemExit("Download da GitHub non riuscito:\n" + (r.stderr.replace(token, "***") if token else r.stderr))
assert os.path.isdir(os.path.join(WORK_DIR, "tracknet_finetune")), "Su GitHub manca tracknet_finetune/: fai prima il push dal PC"

if not os.path.isdir(TNV3_DIR):
    subprocess.run(["git", "clone", "-q", "https://github.com/qaz812345/TrackNetV3.git", TNV3_DIR], check=True)
subprocess.run(["git", "-C", TNV3_DIR, "checkout", "-q", TNV3_COMMIT], check=True)

leggi = lambda p: open(p, encoding="utf-8").read().replace("\r\n", "\n")
assert leggi(os.path.join(TNV3_DIR, "model.py")) == leggi(os.path.join(WORK_DIR, "tracknet3", "model.py")), \
    "Il modello di TrackNetV3 non e' identico a tracknet3/model.py: non proseguire"
print("codice pronto; modello identico a quello del programma")

In [ ]:
!pip install -q parse pycocotools gdown

## 4. Pesi di partenza (quelli del badminton che usa il programma)

In [ ]:
PESI_INIZIALI = os.path.join(WORK_DIR, "tracknet3", "ckpts", "TrackNet_best.pt")
os.makedirs(os.path.dirname(PESI_INIZIALI), exist_ok=True)
shutil.copy2(os.path.join(DRIVE_DIR, "ckpts", "TrackNet_best.pt"), PESI_INIZIALI)   # copy2: tiene la data del file
print("pesi di partenza:", PESI_INIZIALI, f"({os.path.getsize(PESI_INIZIALI) / 1e6:.0f} MB)")

## 5. Dataset pubblico del tennis (TrackNet 2019)

Cerca, in quest'ordine: il dataset già scaricato in questa sessione; una **scorciatoia** (o una copia) dentro `Il mio Drive/rf_coach_vision/datasets`; altrimenti prova a scaricarlo con `gdown`. Se `gdown` non ce la fa (Google limita i download delle cartelle con tanti file), segui le istruzioni che stampa.

Se il dataset è altrove, scrivi il percorso in `ORIGINE_MANUALE` (la cartella che contiene `game1`, `game2`, ...).

In [ ]:
ORIGINE_MANUALE = ""                                  # es. "/content/drive/MyDrive/Dataset"
DATASET_ID = "11r0RUaQHX7I3ANkaYG4jOxXK1OYo01Ut"      # cartella Drive del dataset (README di yastrebksv/TrackNet)
RAW = "/content/tennis_raw"

def label_in(cartella):
    return glob.glob(os.path.join(cartella, "**", "Label.csv"), recursive=True)

def cerca_sul_drive(base, profondita=4):
    """Cartella (anche una scorciatoia) che contiene game1, game2, ... con i Label.csv."""
    if not os.path.isdir(base):
        return None
    for radice, dirs, _ in os.walk(base, followlinks=True):
        if any(d.lower().startswith("game") for d in dirs) and glob.glob(os.path.join(radice, "game*", "*", "Label.csv")):
            return radice
        if radice[len(base):].count(os.sep) >= profondita:
            dirs[:] = []
    return None

ORIGINE = ORIGINE_MANUALE or None
if ORIGINE is None and label_in(RAW):
    ORIGINE = RAW
if ORIGINE is None:
    ORIGINE = cerca_sul_drive(os.path.join(DRIVE_DIR, "datasets"))
if ORIGINE is None:
    print("Provo a scaricare il dataset con gdown ...")
    os.makedirs(RAW, exist_ok=True)
    !gdown --folder {DATASET_ID} -O {RAW} --remaining-ok
    for z in glob.glob(os.path.join(RAW, "**", "*.zip"), recursive=True):
        print("scompatto", z)
        subprocess.run(["unzip", "-q", "-o", z, "-d", os.path.dirname(z)], check=True)
        os.remove(z)
    ORIGINE = RAW if label_in(RAW) else None

if ORIGINE is None:
    print("\nDataset NON trovato. Cosa c'e' in", RAW, ":", os.listdir(RAW) if os.path.isdir(RAW) else "niente")
    print("Apri https://drive.google.com/drive/folders/" + DATASET_ID)
    print("clic destro sul nome della cartella -> Organizza -> Aggiungi scorciatoia -> Il mio Drive/rf_coach_vision/datasets")
    print("poi rilancia questa cella.")
else:
    clip = label_in(ORIGINE)
    partite = sorted({p for p in os.listdir(ORIGINE) if p.lower().startswith("game")})
    print(f"Dataset in {ORIGINE}: {len(clip)} clip, partite {partite}")
    if len(partite) < 10:
        print("ATTENZIONE: mancano delle partite (download incompleto?). Il dataset completo ne ha 10.")

## 6. Prepara il dataset per TrackNetV3

Prende circa `MAX_FRAME` fotogrammi per l'addestramento e tiene le partite `VAL_PARTITE` **solo** per la validazione (divisione per partita, non per fotogramma). Salva i fotogrammi già a 512x288. Con `SALVA_SU_DRIVE = True` mette una copia del risultato su Drive (circa 1,7 GB): la volta dopo, o dopo una disconnessione, si riparte da lì in pochi minuti.

In [ ]:
MAX_FRAME = 10000          # fotogrammi di addestramento (circa)
VAL_PARTITE = "9,10"       # partite usate solo per la validazione
SALVA_SU_DRIVE = True

DATI = "/content/dati_tennis"
os.makedirs(FT_DRIVE, exist_ok=True)
tar_drive = os.path.join(FT_DRIVE, f"dati_tennis_{MAX_FRAME}_val{VAL_PARTITE.replace(',', '-')}.tar")
%cd {WORK_DIR}
if os.path.exists(os.path.join(DATI, "riepilogo_dataset.json")):
    print("dataset gia' pronto in", DATI)
elif os.path.exists(tar_drive):
    print("recupero il dataset pronto dal Drive:", tar_drive)
    subprocess.run(["tar", "-xf", tar_drive, "-C", "/content"], check=True)
else:
    assert ORIGINE, "Manca il dataset pubblico: vedi la cella 5"
    if os.path.isdir(DATI):
        shutil.rmtree(DATI)                      # resto di un tentativo interrotto
    !python tracknet_finetune/prepara_dataset_tennis.py --origine "{ORIGINE}" --uscita {DATI} --max_frame {MAX_FRAME} --val_partite {VAL_PARTITE}
    assert os.path.exists(os.path.join(DATI, "riepilogo_dataset.json")), "Preparazione non riuscita: guarda l'errore qui sopra"
    if SALVA_SU_DRIVE:
        subprocess.run(["tar", "-cf", tar_drive, "-C", "/content", "dati_tennis"], check=True)
        print("copia salvata su Drive:", tar_drive)

riepilogo = json.load(open(os.path.join(DATI, "riepilogo_dataset.json")))
for split, s in riepilogo["split"].items():
    print(f"{split}: {s['fotogrammi']} fotogrammi in {len(s['clip'])} clip, pallina visibile {s['visibili']}, mossa {s['mosse']}")

## 7. Prova veloce (2-3 minuti)

256 campioni, 1 epoca: serve solo a scoprire subito se qualcosa non va (dati, memoria, versioni). Deve finire con `PROVA RIUSCITA`. Scrive in una cartella temporanea, non sul Drive.

In [ ]:
%cd {WORK_DIR}
!python tracknet_finetune/addestra.py --dati {DATI} --tracknetv3 {TNV3_DIR} --pesi_iniziali {PESI_INIZIALI} --uscita /content/prova_ft --prova

## 8. Addestramento

Prima misura i pesi del badminton sulla validazione (il "prima"), poi addestra per `EPOCHE` epoche e dopo ognuna rimisura: tiene la versione migliore. Tutto va su Drive in `tracknet_finetune/<RUN>/`:
- `TrackNet_tennis.pt` (la migliore: è quella da usare),
- `ultimo_completo.pt` (per riprendere),
- `registro.json` (i numeri di ogni epoca).

Per una seconda prova con altre impostazioni cambia `RUN` (es. `run2`): la prima resta.

In [ ]:
RUN = "run1"          # nome della prova
EPOCHE = 15
LR = 1e-4             # passo piccolo: si corregge la rete del badminton, non si riparte da zero
CONGELA = "nessuno"   # "inizio" = tiene fermi i primi blocchi (se la validazione peggiora presto)
RIPRENDI = False      # True = riparte dall'ultima epoca salvata (dopo una disconnessione)

USCITA = os.path.join(FT_DRIVE, RUN)
if os.path.exists(os.path.join(USCITA, "ultimo_completo.pt")) and not RIPRENDI:
    raise SystemExit(f"{USCITA} contiene gia' un addestramento: cambia RUN (es. run2) o metti RIPRENDI = True")
opz_riprendi = "--riprendi" if RIPRENDI else ""
%cd {WORK_DIR}
!python tracknet_finetune/addestra.py --dati {DATI} --tracknetv3 {TNV3_DIR} --pesi_iniziali {PESI_INIZIALI} --uscita "{USCITA}" --epoche {EPOCHE} --lr {LR} --congela {CONGELA} {opz_riprendi}

## 9. Andamento sulla validazione

In [ ]:
import matplotlib.pyplot as plt
reg = json.load(open(os.path.join(USCITA, "registro.json")))
ep = [e["epoca"] for e in reg["epoche"]]
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for k, nome in (("accuracy", "accuratezza"), ("f1", "F1"), ("recall", "richiamo")):
    ax[0].plot(ep, [e[k] for e in reg["epoche"]], marker="o", label=nome)
    if "prima" in reg:
        ax[0].axhline(reg["prima"][k], ls="--", lw=0.8, color=ax[0].lines[-1].get_color())
ax[0].set_title("validazione (tratteggio = pesi del badminton)"); ax[0].set_xlabel("epoca"); ax[0].legend()
ax[1].plot(ep, [e["perdita_train"] for e in reg["epoche"]], marker="o", label="addestramento")
ax[1].plot(ep, [e["perdita_val"] for e in reg["epoche"]], marker="o", label="validazione")
ax[1].set_title("perdita"); ax[1].set_xlabel("epoca"); ax[1].legend()
plt.show()
if "migliore" in reg:
    print("migliore: epoca", reg["migliore"]["epoca"], "accuratezza", round(reg["migliore"]["accuracy"], 3))

## 10. Voto sul set di test (i nostri video, etichettati a mano)

Confronta i pesi del badminton con quelli nuovi sugli **stessi fotogrammi** etichettati, con la previsione fatta **come nel programma** (`tracknet3/predict.py`, senza InpaintNet). Si può lanciare anche **prima** dell'addestramento: misura solo i pesi del badminton.

Per confrontare più prove aggiungi i loro pesi a `PESI`. Le previsioni già fatte restano su Drive (`tracknet_finetune/valutazione/previsioni`) e vengono riusate.

In [ ]:
import pandas as pd
from IPython.display import Markdown, display

RUN = globals().get("RUN", "run1")
PESI = {"badminton": PESI_INIZIALI,
        RUN: os.path.join(FT_DRIVE, RUN, "TrackNet_tennis.pt")}
# PESI["run2"] = os.path.join(FT_DRIVE, "run2", "TrackNet_tennis.pt")

mancano = [n for n, p in PESI.items() if not os.path.exists(p)]
for n in mancano:
    print(f"(salto '{n}': pesi non ancora addestrati)")
    PESI.pop(n)

et_drive = os.path.join(FT_DRIVE, "etichette_test.csv")
et_repo = os.path.join(WORK_DIR, "tracknet_finetune", "test", "etichette_test.csv")
ETICHETTE = et_drive if os.path.exists(et_drive) else et_repo
assert os.path.exists(ETICHETTE), "Manca il set di test: etichettalo sul PC con etichetta_pallina.py"
et = pd.read_csv(ETICHETTE)
print(f"set di test: {ETICHETTE}")
print(f"  {len(et)} etichette: visibile {(et['visibile'] == 1).sum()}, non visibile {(et['visibile'] == 0).sum()}, "
      f"incerta {(et['visibile'] == -1).sum()} (non conta)")

os.makedirs(os.path.join(WORK_DIR, "inputs"), exist_ok=True)
for v in sorted(et["video"].unique()):
    src, dst = os.path.join(DRIVE_DIR, "inputs", v), os.path.join(WORK_DIR, "inputs", v)
    if not os.path.exists(src):
        print("  manca su Drive inputs/:", v, "(le sue etichette vengono saltate)")
    elif not os.path.exists(dst):
        shutil.copy(src, dst)

VAL_LOCALE, VAL_DRIVE = "/content/valutazione", os.path.join(FT_DRIVE, "valutazione")
if os.path.isdir(os.path.join(VAL_DRIVE, "previsioni")):
    shutil.copytree(os.path.join(VAL_DRIVE, "previsioni"), os.path.join(VAL_LOCALE, "previsioni"), dirs_exist_ok=True)

pesi_arg = " ".join(f'"{n}={p}"' for n, p in PESI.items())
%cd {WORK_DIR}
!python tracknet_finetune/valuta_tracknet.py --etichette "{ETICHETTE}" --video_dir inputs --pesi {pesi_arg} --uscita {VAL_LOCALE}

os.makedirs(VAL_DRIVE, exist_ok=True)
for f in glob.glob(os.path.join(VAL_LOCALE, "voto.*")) + glob.glob(os.path.join(VAL_LOCALE, "confronto_*.csv")):
    shutil.copy(f, VAL_DRIVE)
shutil.copytree(os.path.join(VAL_LOCALE, "previsioni"), os.path.join(VAL_DRIVE, "previsioni"), dirs_exist_ok=True)
display(Markdown(open(os.path.join(VAL_LOCALE, "voto.md"), encoding="utf-8").read()))
print("Copia dei risultati su Drive:", VAL_DRIVE)

## Dopo

- Se il voto migliora (soprattutto **richiamo** e **F1** nei gruppi *vicino al colpo* e *pallina mossa*, senza aumentare i **FP2** nel gruppo *a caso*): integrazione nel programma, con il confronto A/B sui video di riferimento.
- Se migliora poco: secondo fine-tuning con i nostri video (etichettati come il set di test, ma su video **diversi** da quelli del test).